# 04 — Econometric Parameter Models Lab
Coeficientes, parámetros interpretables y efectos marginales sobre evidencia real.


In [ ]:
from __future__ import annotations
import sys
from pathlib import Path
import pandas as pd
import numpy as np
cwd=Path.cwd().resolve()
PROJECT_ROOT=cwd if (cwd/"pyproject.toml").exists() else cwd.parent
if not (PROJECT_ROOT/"pyproject.toml").exists():
    raise RuntimeError("Ejecuta este notebook dentro de bd_replica_crm.")
SRC=PROJECT_ROOT/"src"
if str(SRC) not in sys.path:
    sys.path.insert(0,str(SRC))
from replica_cygnus.settings import load_settings
from replica_cygnus.connections import connect_postgres
settings=load_settings(PROJECT_ROOT)
conn=connect_postgres(settings)
def sql_df(sql,params=None):
    return pd.read_sql_query(sql,conn,params=params)
print("DB:",settings.postgres.database)

import statsmodels.formula.api as smf


## 1. Dataset


In [ ]:
data=sql_df("""
SELECT
 decision_at,codigo_proyecto,asesor,canal,medio,
 hour_of_day,day_of_week,is_weekend,
 client_prior_assignments_90d,days_since_previous_assignment,
 project_leads_90d,project_sep_rate_90d,project_minuta_rate_180d,
 advisor_leads_90d,advisor_sep_rate_90d,advisor_minuta_rate_180d,
 global_sep_rate_90d,global_minuta_rate_180d,
 separacion_14d,minuta_60d
FROM features.lead_evidence
WHERE separacion_14d IS NOT NULL OR minuta_60d IS NOT NULL
""")
print("rows:",len(data))


## 2. OLS con errores robustos


In [ ]:
ols_data=data[["minuta_60d","project_sep_rate_90d","advisor_sep_rate_90d","client_prior_assignments_90d","is_weekend"]].dropna()
if len(ols_data)>100:
    ols=smf.ols("minuta_60d ~ project_sep_rate_90d + advisor_sep_rate_90d + client_prior_assignments_90d + is_weekend",data=ols_data).fit(cov_type="HC3")
    print(ols.summary())


## 3. Logit + efectos marginales


In [ ]:
logit_data=data[["separacion_14d","project_sep_rate_90d","advisor_sep_rate_90d","client_prior_assignments_90d","is_weekend"]].dropna()
if len(logit_data)>100 and logit_data["separacion_14d"].nunique()==2:
    logit=smf.logit("separacion_14d ~ project_sep_rate_90d + advisor_sep_rate_90d + client_prior_assignments_90d + is_weekend",data=logit_data).fit(disp=False)
    print(logit.summary())
    print(logit.get_margeff().summary())


## 4. Interacciones


In [ ]:
if len(logit_data)>100 and logit_data["separacion_14d"].nunique()==2:
    inter=smf.logit("separacion_14d ~ project_sep_rate_90d * advisor_sep_rate_90d + client_prior_assignments_90d",data=logit_data).fit(disp=False)
    print(inter.summary())


## 5. Parámetros por proyecto


In [ ]:
rows=[]
for project,g in data.groupby("codigo_proyecto"):
    x=g[["separacion_14d","project_sep_rate_90d","advisor_sep_rate_90d"]].dropna()
    if len(x)>=100 and x["separacion_14d"].nunique()==2:
        try:
            m=smf.logit("separacion_14d ~ project_sep_rate_90d + advisor_sep_rate_90d",data=x).fit(disp=False)
            rows.append({"codigo_proyecto":project,"n":len(x),"beta_project":m.params.get("project_sep_rate_90d"),"beta_advisor":m.params.get("advisor_sep_rate_90d")})
        except Exception:
            pass
project_params=pd.DataFrame(rows)
project_params


## 6. Tabla de parámetros


In [ ]:
if "logit" in globals():
    ci=logit.conf_int()
    params=pd.DataFrame({
        "parameter":logit.params.index,
        "coef":logit.params.values,
        "std_err":logit.bse.values,
        "p_value":logit.pvalues.values,
        "ci_low":ci[0].values,
        "ci_high":ci[1].values
    })
    display(params)


In [ ]:
conn.close(); print("Conexión cerrada.")
